# EM proofreading — Phase A (annotate)

Skeleton-driven review of a whole cell over MICrONS minnie65, dropping tagged
proofreading annotations. **Read-only** here — annotate now, edit manually later
(Phase B), then re-enter on the new root id (Phase C).

**Review model:** `review_next()` builds a **sparse mip1 tube** for the branch — sharp EM +
red target, only the chunks near the skeleton, served from one shared local volume per cell
and cached so revisits are instant. **Play** to glide it sharply *in motion*; **pause** to
drop into the live EM + segmentation and annotate (`m`/`s`/`e`/`q`); `x` to mark the branch
done and advance.

Run in the `em` env (`uv run --extra em jupyter lab`, or the `.venv` kernel); needs a CAVE
token at `~/.cloudvolume/secrets/cave-secret.json`.

## 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)  # silence CloudVolume S3/GCS pool noise

import proofreading.em as em
from proofreading.em.wal import WAL

## 2. Connect to CAVE

`minnie65_public` is the read-only **sandbox** (no edits, no root changes);
`minnie65_phase3_v1` is the live, proofreadable datastack.

In [2]:
client = em.EMClient('minnie65_public')
#   live: em.EMClient('minnie65_phase3_v1', version=<materialization_version>)
print('datastack', client.datastack, '| materialization', client.mat_version)

datastack minnie65_public | materialization 1718


## 3. Start a session

Loads the L2 skeleton, captures the durable **seed supervoxel**, builds the viewer, and
opens/append-resumes the write-ahead log.

In [3]:
root_id = 864691135572530981   # example cell on minnie65_public
sess = em.ProofreadSession(
    client, root_id, wal_dir='./proofread_sessions',
    step_nm=1000.0,                 # camera node spacing along each branch
    tube_mip=1,                     # glide resolution (1 = 16 nm); the default review mode
    tube_radius_nm=1000.0,          # tube radius around the skeleton (wider = more context + data)
    orient_to_path=False,           # axis-aligned sections (fast); True = cross-section ⊥ neurite
    seconds_per_step=0.4,           # glide speed (seconds per node)
    cross_section_render_scale=1.0, # live layers (shown on pause) at full res
)
print('seed', sess.seed, '| branch paths', len(sess.tree.branch_paths), '|', sess.summary())

seed 111692652428576376 | branch paths 189 | {'to_review': 179, 'covered': 8, 'omitted': 2}


## 4. Open the viewer

Open the URL in a browser — 4-panel layout (3 cross-sections + 3D). The target cell is
highlighted; reveal neighbors with `n`.

In [4]:
sess.viewer

http://localhost:50031/v/3aa9f03eaf5ddc449cbc727125870fe7db9e1c80/

## 5. Review — glide, pause, annotate

`review_next()` builds the next to-review branch's **sparse mip1 tube** (sharp EM + red
target; ~10–30 s the first time, instant once cached) and starts a **paused** glide.

- **`fly.play()`** → continuous sharp glide along the neurite (the local tube renders in motion).
- **`fly.pause()`** → live full-res EM + real segmentation paint at that spot; annotate.
- **`fly.reverse()` / `fly.step(1)`** → manual control.

Keys in the neuroglancer window (while paused):

| key | action |
|-----|--------|
| `m` | merge error |
| `s` | split error |
| `e` | extend |
| `q` | question |
| `n` | toggle the segment under the cursor (reveal/hide a neighbor) |
| `x` | mark the current branch reviewed (and advance to the next) |

A `merge error` ends the branch early and **prunes its distal subtree** off the checklist.
Annotations record the click `xyz` immediately to the durable WAL.

_Fallbacks:_ `sess.review_path(pid, mode='live')` (no precompute) or `mode='preview'` (coarse in-memory).

In [ ]:
fly = sess.review_next()    # builds the branch tube (sharp mip1 EM + red target) + PAUSED glide
fly

In [ ]:
fly.play()     # sharp glide along the neurite

In [ ]:
fly.pause()    # live img+seg paint here -> annotate (m/s/e/q), then fly.play() to resume, or x to advance

## 6. Control panel (optional)

Branch-path checklist + **Review** / **Mark done** / **Resolve supervoxels**, with the
FlyThrough play/pause/step controls. The controls + checklist follow the current branch —
**Mark done** (or the `x` key) advances to the next branch automatically.

In [5]:
sess.panel()

## 7. Checkpoint — resolve supervoxels

Annotations store the click `xyz` immediately (durable); supervoxels are derived in batch
from CloudVolume. Run at checkpoints / before stepping away.

In [ ]:
print('resolved', sess.resolve_supervoxels(), 'supervoxels')

## 8. Inspect the log

The append-only write-ahead log is the source of truth — replay it any time.

In [ ]:
state = WAL.load(sess.wal.path)
print('log:', sess.wal.path)
for a in state.annotations.values():
    print(f'  {a.tag:12s} xyz={[round(c) for c in a.xyz]} supervoxel={a.supervoxel}')
print('coverage:', sess.summary())

## 9. After edits — re-entry (Phase C)

Once you've performed the manual splits/merges the root id changes. Recover the current
root from the durable seed and start fresh: the same WAL resumes, prior coverage re-attaches
by L2 id, and edited regions fall back to `to_review`.

```python
new_root = client.current_root(sess.seed)
sess2 = em.ProofreadSession(client, new_root, wal_dir='./proofread_sessions')
sess2.summary()
```

## 10. Shutdown

In [ ]:
sess.close()

## Appendix — validate local-layer-in-motion (optional, one-time)

The review model assumes a **local** layer renders sharp *during* camera motion (unlike the
live graphene seg). Confirm it once in a **fresh kernel** (this starts its own neuroglancer
server): run the cell, open the viewer, `spike_fly.play()`, and watch the synthetic
checkerboard while it moves. Then restart the kernel for the real workflow.

In [ ]:
spike_viewer, spike_fly = em.localvolume_spike()   # synthetic; no CAVE needed
spike_viewer

In [ ]:
spike_fly.play()    # watch the checker while it moves; spike_fly.pause() to stop